In [ ]:
from wekeo_iasi_l3.download import get_IASI_products
from wekeo_iasi_l3.reader_l2.reader import read_iasi_l2
from wekeo_iasi_l3.global_accumulator import GlobalAccumulator2D

import xarray as xr

from wekeo_iasi_l3.stubs import log
from wekeo_iasi_l3.stubs import env
from datetime import date
    
def  get_iasi_l3(day: date) -> xr.Dataset:
    """Download and Compile multiple FRP files into a single xarray Dataset."""
    
    log.info(f"Downloading FRP products for date: {day}")
    
    eday = day.strftime("%Y-%m-%d")
    sday = day.strftime("%Y-%m-%d") # same day for single day query

    files = get_IASI_products(
        start_date = sday,
        end_date = eday,
    )

    # Read specific variables
    variables = [
        'INTEGRATED_CO',
    ]
    
    # files = files[:3]  # DEBUG: limit to 3 files for demo purposes
    
    # compute the reprojected L3 dataset
    PROPER_RESOLUTION_WIDTH = 3272  # ~1km at the equator, adjust as needed
    SIMPLER_RESOLUTION_WIDTH = 1000  # for demo purposes
    WIDTH = SIMPLER_RESOLUTION_WIDTH

    acc = GlobalAccumulator2D(width = WIDTH, height = int(WIDTH/2))
    ds = xr.Dataset()
    
    for variable in variables:
        
        log.info(f"Processing variable: {variable}")
        
        for idx, file in enumerate(files):
            log.info(f"Reading file {idx + 1}/{len(files)}: {file} ")
            dataset = read_iasi_l2(file, variables=variables)
            acc.add(dataset[variable].values, lat=dataset.latitude.values, lon=dataset.longitude.values)
        
        # append the gridded L3 to the output dataset
        ds[variable + "_MEAN"] = acc.get_mean_data_array()
        ds[variable + "_CNT"] =  acc.get_cnt_data_array()
        
        if "unit" in dataset[variable].attrs: 
            ds[variable + "_MEAN"].attrs["unit"] = dataset[variable].attrs["unit"]
            
        if "long_name" in dataset[variable].attrs: 
            ds[variable + "_MEAN"].attrs["long_name"] = dataset[variable].attrs["long_name"] + " gridded mean"

    ds.attrs["description"] = f"Gridded L3 dataset from IASI Level 2 products for date: {day}"
    ds.attrs["source_files"] = ", ".join(f.name for f in files)

    return ds    


ds_lvl3 = get_iasi_l3(date(2024, 8, 15))
print(ds_lvl3)

In [ ]:
# Plot L3 gridded data

from wekeo_iasi_l3.plot_L3_IASI import plot_L3_IASI
from pathlib import Path

colormap = "viridis"
figsize = (14, 7)

figdir = Path("figures")
figdir.mkdir(parents=False, exist_ok=True)
# figdir = None  # Disables figure saving

# Plot INTEGRATED_CO variables
plot_L3_IASI(ds_lvl3, variable='INTEGRATED_CO_MEAN', cmap=colormap, figsize=figsize, save_fig_dir=figdir)
plot_L3_IASI(ds_lvl3, variable='INTEGRATED_CO_CNT', cmap='plasma', figsize=figsize, save_fig_dir=figdir)